# Within- vs pan-stratum models (cancer type and treatment class)

Drives the two `pipelines.trajectories.*` comparison scripts, each as a subprocess so no state
leaks between them:

| Script | Stratum | Output dir |
|---|---|---|
| `within_vs_pan_cancer_models` | `CANCER_TYPE` | `results/pan_vs_within_cancer/` |
| `within_treatment_vs_pan_treatment_models` | first-line `TREATMENT_CLASSIFICATION` | `results/pan_vs_within_treatment/` |

Runs **after** [03_build_prediction_datasets.ipynb](03_build_prediction_datasets.ipynb) and
**before** [06b_generate_figure_data.ipynb](06b_generate_figure_data.ipynb) — `figures.prep.figure2`
reads `metrics_by_cancer_type.csv` and `metrics_by_treatment.csv` to build
`fig2_within_vs_pan_cancer.csv` / `fig2_within_vs_pan_treatment.csv`.

Both scripts fit their own models from the `icd3_post` embedding prediction dataset and the
non-text covariates, so they do **not** consume the full-cohort risk scores from 04b/05 and do not
have to wait on the SLURM arrays. `RESULTS_PATH` is a write target here, not a read.

The other `pipelines.trajectories.*` entry point, landmark mortality trajectories, has its own
driver in [04d_run_mortality_trajectories.ipynb](04d_run_mortality_trajectories.ipynb).

### What the comparison asks

Does a model fit *within* one cancer type (or one first-line treatment class) beat a pan-stratum
model on that stratum's held-out patients? Both scripts split 75/25 on `DFCI_MRN` (seed 1234) from
the `icd3_post` embedding prediction dataset, fit a within-stratum Coxnet per eligible stratum, and
score the held-out patients with both arms.

The pan arm is **size-matched**: each stratum's pan comparator is fit on a random train subset of
the same N with that stratum's own patients excluded, rather than one full-cohort model. Without
that, the pan arm would win on training-set size alone and the comparison would say nothing about
whether stratum-specific signal exists.

### Strata floors (module constants in each script — change them there, not here)

- `MIN_STRATUM_N = 500` — cancer: rarer types collapse into `OTHER` before modeling; treatment:
  the class gets no within-model and no matched pan model, so it never enters the comparison.
- `MIN_TRAIN_N = 100` — floor on the actual train-split subset before a within-model is fit.
- `MIN_HELDOUT_N = 30` — floor for entering the per-stratum held-out table (and the figure);
  below it AUC/C-index are too unstable to plot.

The rarity map is learned on **training data only** and applied unchanged to held-out patients, so
held-out category frequencies never shape features.

### Resume

Both scripts checkpoint per stratum under `<outdir>/checkpoints/` with `RESUME = True`. Each run
fingerprints its deterministic inputs (cohort sizes, strata list, seed, a hash of the modeling
columns); if the fingerprint moved, the stored checkpoints are stale and `RunCheckpoint` starts
fresh on its own. So an interrupted run can simply be re-executed, and a cohort rebuild upstream
does not silently reuse old fits.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import config
import schemes

RESULTS_PATH = config.RESULTS_PATH
FEATURE_PATH = config.FEATURE_PATH

# The scheme both scripts load (`load_embedding_prediction_df("icd3_post")`), mirrored here for
# the precondition check. Change it in the scripts, not here.
SCHEME = "icd3_post"
EMBEDDING_FILE = os.path.join(config.SURV_PATH, schemes.embedding_file(SCHEME))

# (label, module, output dir, per-stratum metrics filename, stratum column)
RUNS = [
    ("cancer",    "pipelines.trajectories.within_vs_pan_cancer_models",
     os.path.join(RESULTS_PATH, "pan_vs_within_cancer"),    "metrics_by_cancer_type.csv", "CANCER_TYPE"),
    ("treatment", "pipelines.trajectories.within_treatment_vs_pan_treatment_models",
     os.path.join(RESULTS_PATH, "pan_vs_within_treatment"), "metrics_by_treatment.csv",   "TREATMENT"),
]

# --- Run toggles: turn off a comparison you do not need to re-execute ---
RUN_CANCER = True
RUN_TREATMENT = True

# Skip a run whose metrics file already exists. Both scripts resume from their own checkpoints,
# so leaving this False and re-running an interrupted comparison is cheap — it refits only the
# strata that never completed.
SKIP_IF_DONE = False

print(f"v2 root:      {V2_ROOT}")
print(f"results path: {RESULTS_PATH}")
print(f"scheme:       {SCHEME}")

## Preconditions

Both scripts read the same two inputs; the treatment script needs one more. A missing file here is
the difference between "the run failed" and "the run never had its inputs" — worth knowing before
a long fit starts.

In [ ]:
PRECONDITIONS = [
    (f"{SCHEME} embeddings", EMBEDDING_FILE,                                             "cancer, treatment"),
    ("cancer types",         os.path.join(FEATURE_PATH, "cancer_type_df.csv.gz"),         "cancer, treatment"),
    ("treatment by line",    os.path.join(FEATURE_PATH, "categorical_treatment_data_by_line.csv.gz"), "treatment"),
]

missing = []
for label, path, used_by in PRECONDITIONS:
    ok = os.path.exists(path)
    if not ok:
        missing.append(label)
    print(f"[{'ok ' if ok else 'MISSING'}] {label:<20} (used by {used_by:<19}) {path}")

if missing:
    print(f"\n{len(missing)} input(s) missing: {', '.join(missing)}")
    print("Runs depending on them will fail. This cell does not raise — inspect and decide.")
else:
    print("\nAll inputs present.")

## Run

Each comparison is `python -m <module>` with `cwd` set to `v2/`. Output streams straight through —
these are long runs behind per-stratum progress, so silence would be indistinguishable from a hang.

A failure does **not** stop the other comparison: the two scripts share inputs but neither reads the
other's output, so a broken cancer run says nothing about the treatment run. Both are reported at
the end.

Interrupting this cell leaves the completed strata on disk under `<outdir>/checkpoints/`; re-running
picks up from there.

In [ ]:
def _metrics_path(outdir: str, fname: str) -> str:
    return os.path.join(outdir, fname)


ENABLED = {"cancer": RUN_CANCER, "treatment": RUN_TREATMENT}

results: list[tuple[str, str, float]] = []  # (label, status, elapsed seconds)

for label, module, outdir, fname, _stratum_col in RUNS:
    if not ENABLED[label]:
        print(f"\n=== {label}: disabled (toggle off) ===")
        results.append((label, "disabled", 0.0))
        continue

    if SKIP_IF_DONE and os.path.exists(_metrics_path(outdir, fname)):
        print(f"\n=== {label}: already done ({fname} present), skipping ===")
        results.append((label, "skipped", 0.0))
        continue

    print(f"\n{'=' * 78}\n=== {label}: python -m {module}\n{'=' * 78}", flush=True)
    started = time.time()
    proc = subprocess.run([sys.executable, "-m", module], cwd=str(V2_ROOT))
    elapsed = time.time() - started

    if proc.returncode == 0:
        status = "ok"
        print(f"\n[{label}] finished in {elapsed / 60:.1f} min")
    else:
        status = f"FAILED (exit {proc.returncode})"
        print(f"\n[{label}] FAILED with exit code {proc.returncode} after {elapsed / 60:.1f} min")
    results.append((label, status, elapsed))

print(f"\n{'=' * 78}\n=== Run summary ===")
for label, status, elapsed in results:
    suffix = f"  ({elapsed / 60:.1f} min)" if elapsed else ""
    print(f"  {label:<10} {status}{suffix}")

failed = [label for label, status, _ in results if status.startswith("FAILED")]
if failed:
    print(f"\n{len(failed)} run(s) failed: {', '.join(failed)} — "
          "the sections below will show whatever landed on disk.")

## Output inventory

What is actually on disk now, so a partial or interrupted run stays legible. Each run writes three
CSVs plus a checkpoint directory; the checkpoint count is the number of strata that completed.

In [ ]:
OUTPUT_FILES = ["train_risk_scores.csv", "held_out_risk_scores.csv"]

for label, _module, outdir, fname, _stratum_col in RUNS:
    print(f"\n{label}  ({outdir})")
    if not os.path.isdir(outdir):
        print("  [none] no output directory — this comparison has never completed a run")
        continue

    for f in OUTPUT_FILES + [fname]:
        path = os.path.join(outdir, f)
        if os.path.exists(path):
            size_mb = os.path.getsize(path) / 1e6
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
            print(f"  [ok     ] {f:<28} {size_mb:>8.1f} MB   {mtime}")
        else:
            print(f"  [missing] {f}")

    ckpt_dir = os.path.join(outdir, "checkpoints")
    if os.path.isdir(ckpt_dir):
        n_ckpt = len([n for n in os.listdir(ckpt_dir) if not n.startswith(".")])
        log_path = os.path.join(ckpt_dir, "progress.log")
        log_note = "" if os.path.exists(log_path) else "  (no progress.log)"
        print(f"  [ok     ] {'checkpoints/':<28} {n_ckpt:>8} file(s){log_note}")
    else:
        print(f"  [missing] checkpoints/")

## Results

The per-stratum held-out comparison each script writes. `DELTA_AUC_WITHIN_MINUS_PAN > 0` means the
within-stratum model beat its size-matched pan comparator on that stratum's held-out patients; the
table is sorted by that delta, with the `Overall` row (the whole held-out set, not a stratum)
prepended as the reference line.

Both metrics are held-out: `CINDEX_*` is Harrell's C, `AUC_*` is the mean of `cumulative_dynamic_auc`
over the evaluation grid. Every read is guarded, so this section is safe to run on a partial pipeline.

In [ ]:
import polars as pl

METRIC_COLS = ["CINDEX_PAN", "CINDEX_WITHIN", "AUC_PAN", "AUC_WITHIN",
               "DELTA_AUC_WITHIN_MINUS_PAN", "DELTA_WITHIN_MINUS_PAN"]

with pl.Config(tbl_rows=60, tbl_cols=12, tbl_width_chars=160, float_precision=3):
    for label, _module, outdir, fname, stratum_col in RUNS:
        path = os.path.join(outdir, fname)
        print(f"\n{'=' * 78}\n=== {label}: {fname}\n{'=' * 78}")
        if not os.path.exists(path):
            print("  not written — run has not completed")
            continue

        metrics = pl.read_csv(path)
        overall = metrics.filter(pl.col(stratum_col) == "Overall")
        strata = metrics.filter(pl.col(stratum_col) != "Overall")

        if overall.height:
            row = overall.row(0, named=True)
            print(f"Overall held-out (n={row['N_HELDOUT']:,}): "
                  f"AUC pan={row['AUC_PAN']:.3f} within={row['AUC_WITHIN']:.3f} "
                  f"(delta {row['DELTA_AUC_WITHIN_MINUS_PAN']:+.3f})   |   "
                  f"C-index pan={row['CINDEX_PAN']:.3f} within={row['CINDEX_WITHIN']:.3f} "
                  f"(delta {row['DELTA_WITHIN_MINUS_PAN']:+.3f})")

        n_within_wins = int((strata["DELTA_AUC_WITHIN_MINUS_PAN"] > 0).sum())
        print(f"{strata.height} strata compared; within-stratum model wins on AUC in "
              f"{n_within_wins}/{strata.height}\n")

        print(metrics.select([stratum_col, "N_HELDOUT"] + [c for c in METRIC_COLS if c in metrics.columns]))

## Hand-off to 06b

`figures.prep.figure2` reads both metrics files and writes `fig2_within_vs_pan_cancer.csv` and
`fig2_within_vs_pan_treatment.csv` into `FIGURE_DATA_DIR`. This cell confirms 06b has what it
needs — a missing metrics file here means Figure 2's within-vs-pan panels will be absent or
stale, not that 06b will fail loudly.

In [ ]:
ready = True
for label, _module, outdir, fname, _stratum_col in RUNS:
    path = os.path.join(outdir, fname)
    ok = os.path.exists(path)
    ready &= ok
    print(f"[{'ok ' if ok else 'MISSING'}] figure2 input ({label:<9}) {path}")

print(f"\nfigure data dir: {config.FIGURE_DATA_DIR}")
for csv in ["fig2_within_vs_pan_cancer.csv", "fig2_within_vs_pan_treatment.csv"]:
    fig_path = os.path.join(config.FIGURE_DATA_DIR, csv)
    if os.path.exists(fig_path):
        mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(fig_path)))
        print(f"  [existing] {csv:<36} last written {mtime}")
    else:
        print(f"  [none    ] {csv:<36} (06b has not written it yet)")

print("\n" + ("Ready for 06b_generate_figure_data.ipynb."
               if ready else
               "Not ready — re-run the missing comparison above before 06b."))